<a href="https://colab.research.google.com/github/swrobuts/dav/blob/main/notebooks/04_Daten_Inspizieren.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Notebook 04: Daten inspizieren & Explorative Datenanalyse (EDA)

**Szenario**: Du hast einen neuen Datensatz erhalten (flights.csv) – was machst du als erstes?

**Lernziele:**
- df.info(), df.describe(), df.dtypes verstehen
- Fehlende Werte mit missingno visualisieren
- Verteilungen (Histogramm, Boxplot, Violinplot)
- Kategoriale Analysen
- Ausreißer mit Z-Score und IQR erkennen
- Korrelationsanalyse

---

In [ ]:
# ── Setup: Pakete installieren (nur in Colab nötig) ──────────────────────────
import subprocess, sys
packages = ['ydata-profiling', 'missingno', 'plotly', 'kaleido']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print("Setup abgeschlossen!")


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/swrobuts/dav/main/data/"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Stil setzen
sns.set_theme(style='whitegrid', palette='Blues')
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv(BASE_URL + "flights.csv")
print(f'Dataset geladen: {df.shape[0]:,} Flüge, {df.shape[1]} Spalten')

## 1️⃣ Erster Blick: head(), tail(), info(), describe()

In [ ]:
print('=== df.head(5) ===')
print(df.head(5))
print()
print('=== df.dtypes ===')
print(df.dtypes)
print()
print('=== df.info() ===')
df.info()

In [ ]:
print('=== df.describe() – Numerische Spalten ===')
print(df.describe().round(2))
print()
print('=== df.describe(include="object") – Kategorische Spalten ===')
print(df.describe(include='object'))

## 2️⃣ Fehlende Werte analysieren

In [ ]:
print('=== Fehlende Werte (absolut) ===')
print(df.isnull().sum())
print()
print('=== Fehlende Werte (prozentual) ===')
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

In [ ]:
try:
    import missingno as msno
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    msno.matrix(df, ax=axes[0], color=(0.22, 0.54, 0.81), sparkline=False)
    axes[0].set_title('Missing Values Matrix', fontsize=12, fontweight='bold')
    msno.bar(df, ax=axes[1], color='#E8792F')
    axes[1].set_title('Datenvollständigkeit je Spalte', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('missingno nicht verfügbar – Alternative:')
    missing = df.isnull().sum()
    missing[missing > 0].plot(kind='bar', color='#E8792F', figsize=(10,4))
    plt.title('Fehlende Werte je Spalte')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 3️⃣ Verteilungsanalyse

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramm: Passengers
axes[0].hist(df['passengers'], bins=20, color='#0389CF', alpha=0.7, edgecolor='white')
axes[0].set_title('Passagierzahlen – Histogramm', fontweight='bold')
axes[0].set_xlabel('Anzahl Passagiere')
axes[0].set_ylabel('Häufigkeit')
axes[0].axvline(df['passengers'].mean(), color='#E8792F', linestyle='--', label=f'Mean: {df["passengers"].mean():.1f}')
axes[0].legend()

# Boxplot: Passengers with mean marker
sns.boxplot(x=df['passengers'], ax=axes[1], color='#0389CF', showmeans=True,
            meanprops={"marker":"x", "markeredgecolor":"white", "markersize":"10"})
axes[1].set_title('Passagierzahlen – Boxplot (x = Mittelwert)', fontweight='bold')
axes[1].set_xlabel('Anzahl Passagiere')

plt.tight_layout()
plt.show()

## 4️⃣ Ausreißer erkennen: Z-Score & IQR

In [ ]:
from scipy import stats

data_col = df['passengers']

# Z-Score
z_scores = np.abs(stats.zscore(data_col))
outliers_z = data_col[z_scores > 3]
print(f'Z-Score Ausreißer (|z|>3): {len(outliers_z):,} ({len(outliers_z)/len(data_col)*100:.1f}%)')

# IQR
Q1 = data_col.quantile(0.25)
Q3 = data_col.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers_iqr = data_col[(data_col < lower) | (data_col > upper)]
print(f'\nIQR Ausreißer: {len(outliers_iqr):,} ({len(outliers_iqr)/len(data_col)*100:.1f}%)')
print(f'  Grenzen: [{lower:.1f}, {upper:.1f}] Passagiere')

## 5️⃣ Korrelationsanalyse

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Benutzerdefinierte Blau-Skala von Mittelblau zu Navy
colors = ['#3498db', '#2980b9', '#1a5276', '#001f3f']
custom_blue_cmap = LinearSegmentedColormap.from_list('custom_blue', colors, N=100)

numeric_df = df.select_dtypes(include='number').dropna()
corr = numeric_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap=custom_blue_cmap,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Korrelationsmatrix (Jahr vs. Passagiere)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## ✅ Challenges

1. Welche Airline hat die höchste durchschnittliche Ankunftsverspätung?
2. Gibt es einen Unterschied zwischen Montags- und Freitagsflügen bei der Verspätung?
3. Visualisiere die Verteilung der Fluglänge (distance) als Histogramm

In [ ]:
# Adaptierte Challenges f  r den vorhandenen Datensatz:

# Challenge 1: In welchem Jahr gab es die h&#246;chste durchschnittliche Passagierzahl?
avg_per_year = df.groupby('year')['passengers'].mean().sort_values(ascending=False)
print(f"H&#246;chste Passagierzahl im Schnitt: {avg_per_year.idxmax()} ({avg_per_year.max():.1f})")

# Challenge 2: Berechne das Wachstum der Passagiere von 1949 bis 1960
growth = df[df['year'] == 1960]['passengers'].sum() - df[df['year'] == 1949]['passengers'].sum()
print(f"Absolutes Wachstum: {growth} Passagiere")

# Challenge 3: Visualisiere die Passagierzahlen   ber die Monate (Saisonalit&#228;t)
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='month', y='passengers', hue='year', palette='viridis')
plt.title('Passagiere je Monat (Saisonalit&#228;t)')
plt.xticks(rotation=45)
plt.show()

---
**Weiter mit:** `05_Daten_Bereinigen.ipynb`